# Lab 4 - Hugging Face Hub and CLI (Python companion notebook)

**Day 2 - Tools and Ecosystems for LLM Agents** | Use this notebook alongside the **Lab 4 README**

Most of Lab 4 happens in the **terminal** with the `hf` command-line tool. This notebook covers the steps that are easier in Python:

* **Part 1:** check your login and explore the Hub with the Python API (`HfApi`)
* **Part 2:** prepare your Day 1 model for publishing and write its **model card**
* **Part 3:** after you upload it with the CLI, load it back **from the Hub** and inspect its commit history

> Do README **Tasks 1-3** (account, token, `hf auth login`) before running Part 1.

In [ ]:
# Setup
import os, sys, json, shutil, warnings
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
warnings.filterwarnings("ignore")
from pathlib import Path
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

import huggingface_hub
from huggingface_hub import HfApi
api = HfApi()
print("huggingface_hub", huggingface_hub.__version__)

---
## Part 1 - Who am I, and what's on the Hub?

`hf auth login` saved your token on this VM, so the Python API uses it automatically. There's nothing to paste here.

In [ ]:
try:
    me = api.whoami()
    HF_USER = me["name"]
    token_role = me.get("auth", {}).get("accessToken", {}).get("role", "?")
    print(f"Logged in as: {HF_USER}  (token permission: {token_role})")
except Exception as e:
    HF_USER = None
    print("Not logged in. Run `hf auth login` in the terminal (README Task 2), then run this cell again.")
    print("Details:", type(e).__name__, str(e)[:200])

The Hub hosts over two million models. `list_models` searches them, just like `hf models ls` in the terminal.

In [ ]:
print("Most-downloaded text-classification models for 'sentiment':")
for m in api.list_models(search="sentiment", pipeline_tag="text-classification", sort="downloads", limit=5):
    print(f"  {m.downloads:>12,}  {m.id}")

info = api.model_info("distilbert/distilbert-base-uncased", files_metadata=True)
print(f"\n{info.id}: task={info.pipeline_tag}, downloads={info.downloads:,}, likes={info.likes}")
for f in info.siblings:
    if f.size:
        print(f"  {f.size / 1e6:9.1f} MB  {f.rfilename}")

A model repo is a Git repository of files: weights (`model.safetensors`), configuration (`config.json`),
tokenizer files, and a `README.md` that the Hub renders as the **model card**.

---
## Part 2 - Prepare your Day 1 model for publishing

In Day 1 Lab 2 you fine-tuned DistilBERT on SST-2 and saved it. The cell below finds it.
If you didn't finish Lab 2 (or are on a different VM), it falls back to a public DistilBERT sentiment model so you can still
practise publishing; the model card then says so honestly.

In [ ]:
LAB2 = Path("../../day1/lab02-transfer-learning/outputs")
candidates = [LAB2 / "full" / "final", LAB2 / "full_2000" / "final", LAB2 / "frozen" / "final", LAB2 / "frozen_2000" / "final"]
source = next((c for c in candidates if (c / "config.json").exists()), None)

PUBLISH_DIR = Path("outputs/model-to-publish")
if PUBLISH_DIR.exists():
    shutil.rmtree(PUBLISH_DIR)

if source:
    shutil.copytree(source, PUBLISH_DIR, ignore=shutil.ignore_patterns("training_args.bin", "checkpoint-*"))
    FROM_DAY1 = True
    print("Found your Day 1 model:", source.resolve())
else:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    fallback = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
    AutoModelForSequenceClassification.from_pretrained(fallback).save_pretrained(PUBLISH_DIR)
    AutoTokenizer.from_pretrained(fallback).save_pretrained(PUBLISH_DIR)
    FROM_DAY1 = False
    print("No Day 1 model found, so using a copy of", fallback, "for practice.")

for f in sorted(PUBLISH_DIR.iterdir()):
    print(f"  {f.stat().st_size / 1e6:8.1f} MB  {f.name}")

### Write the model card

A model with no card is a model nobody can trust. The card has two parts:

* **YAML metadata** at the top: license, base model, dataset, metrics, task. The Hub uses it for search, filters and the widget.
* **Markdown** below: what the model does, how it was trained, how well it works, and its **limitations**.

In [ ]:
from huggingface_hub import ModelCard, ModelCardData, EvalResult

# Read the accuracy you measured in Day 1 (results.json), if it exists
results_file = LAB2 / "results.json"
day1 = json.loads(results_file.read_text()) if results_file.exists() else {}
run_name = source.parent.name if source else None          # e.g. "full" or "frozen_2000": the run you're publishing
acc = day1.get(run_name, {}).get("accuracy") if FROM_DAY1 else None

REPO_NAME = "distilbert-sst2-lab"
REPO_ID = f"{HF_USER}/{REPO_NAME}" if HF_USER else f"<your-username>/{REPO_NAME}"

card_data = ModelCardData(
    language="en",
    license="apache-2.0",
    base_model="distilbert/distilbert-base-uncased" if FROM_DAY1 else "distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    datasets=["stanfordnlp/sst2"],
    pipeline_tag="text-classification",
    library_name="transformers",
    tags=["sentiment-analysis", "training-lab"],
    model_name=REPO_NAME,
    eval_results=[EvalResult(task_type="text-classification", dataset_type="stanfordnlp/sst2", dataset_name="SST-2",
                             dataset_split="validation", metric_type="accuracy", metric_value=acc)] if acc else None,
)

origin = ("Fine-tuned in the *Agentic AI with Copilot* training, Day 1 Lab 2, on 2,000 SST-2 sentences on a CPU."
          if FROM_DAY1 else
          "**Practice copy** of `distilbert/distilbert-base-uncased-finetuned-sst-2-english`, published to learn the Hub workflow.")
score = f"**{acc:.1%}** accuracy on the SST-2 validation set (872 sentences)." if acc else "Not evaluated in this lab."

content = f"""---
{card_data.to_yaml()}
---

# {REPO_NAME}

A DistilBERT model that classifies English sentences as **POSITIVE** or **NEGATIVE**.

## Origin
{origin}

## How to use
```python
from transformers import pipeline
clf = pipeline("text-classification", model="{REPO_ID}")
clf("The food was great but the service was slow.")
```

## Results
{score}

## Limitations
* Trained on short movie-review sentences; it may be unreliable on other domains (support tickets, tweets, code).
* English only. Misreads negation and sarcasm, for example *"Not bad at all"*.
* A training exercise, not for production use.
"""
card = ModelCard(content)
card.save(PUBLISH_DIR / "README.md")
print(card.content[:900])

> **Edit me:** open `outputs/model-to-publish/README.md` in VS Code and add a sentence of your own under *Limitations*.

Now go back to the README and do **Task 4**: create the repo and upload this folder with the `hf` CLI.

---
## Part 3 - Load your model back from the Hub

After README Task 4, your model lives on the Hub. Anyone with access, on any machine, can now use it by its **repo id**.
`pipeline` downloads it into the local cache the first time.

In [ ]:
from transformers import pipeline

if not HF_USER:
    raise SystemExit("Log in first (README Task 2), then re-run Part 1 and Part 2.")

print("Loading", REPO_ID, "from the Hub...")
clf = pipeline("text-classification", model=REPO_ID)          # private repos use your saved login
for text in ["The food was great but the service was slow.",
             "Absolutely loved the new update!",
             "Not bad at all."]:
    r = clf(text)[0]
    print(f"  {r['label']:<8} {r['score']:.3f}  {text}")

In [ ]:
# Every upload is a Git commit. List the history of your repo:
if not HF_USER:
    raise SystemExit("Log in first (README Task 2).")
for c in api.list_repo_commits(REPO_ID):
    print(f"{c.created_at:%Y-%m-%d %H:%M}  {c.commit_id[:8]}  {c.title}")

print("\nTags:", [t.name for t in api.list_repo_refs(REPO_ID).tags])
print("Open it in the browser: https://huggingface.co/" + REPO_ID)

After README **Task 6** (the Git interface), run the commit-history cell again: your `git push` appears as a new commit.

**Use it in Lab 5:** set `MY_REPO = "<your-username>/distilbert-sst2-lab"` in Lab 5, Part 1.

---
## Summary

| You did | Why it matters |
|---|---|
| Logged in with a token, explored the Hub | Tokens give scoped access; the Hub is searchable from code |
| Wrote a model card with metadata | Documentation and limitations are part of a responsible release |
| Uploaded with the CLI, tagged a version | Models are versioned like code; tags pin exact releases |
| Loaded your model by repo id | Share once, use anywhere |